# Wire and smoke-test the SFT trainer

With QLoRA proven (`qlora_setup.ipynb`) and the masking strategy locked (prompt-completion → completion-only loss - `probe_template.ipynb`), this notebook assembles the trainer and runs a short smoke test.

The goal is **not** a trained model — it is to catch template, masking, and OOM bugs in *minutes* before committing to a full run. Three things must come out green:
1. the **loss** is finite and trends down over ~30 steps,
2. the **label mask** is `-100` on the prompt and real tokens on the completion (token-level proof),
3. peak VRAM **fits in 12 GB**.

Inputs already decided: `to_prompt_completion` (the reshape), `max_length=1024` (from the length probe), and the 4-bit + LoRA model.

In [1]:
from pathlib import Path
from typing import cast

import datasets
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
import tool_forge
from tool_forge.dataset import render_prompt_completion
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

/home/sjullens/Code/Projects/LLM_structured_response/.venv-train/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load and reshape the dataset
Load `data/train.jsonl` and map it to the prompt-completion view TRL masks on — columns `prompt`, `completion`, `tools`.

- a small slice is enough for a smoke test ~256
- Arrow schema inference chokes on the heterogeneous nested `tools`, fall back to reading the lines with (`json.loads`) and load using `Dataset.from_list`

In [2]:
jsonl = tool_forge.dataset.load_dataset(Path("../data/train.jsonl"))

In [3]:
print(type(jsonl), len(jsonl))

<class 'list'> 46893


In [8]:
tokenizer = AutoTokenizer.from_pretrained(tool_forge.QWEN_4B_INSTRUCT)
# Pre-render -> two text columns. Arrow stores strings trivially; no nested JSON to infer.
dataset = datasets.Dataset.from_list(
    [render_prompt_completion(r, tokenizer) for r in jsonl[:256]])
dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 256
})

## Build the 4-bit + LoRA model
Same QLoRA build as the previous notebook: nf4 `BitsAndBytesConfig` → `from_pretrained` → `prepare_model_for_kbit_training` → `LoraConfig`.

Either build it here, **or** pass the base model id + `peft_config=lora_config` to `SFTTrainer` and let it run `prepare_model_for_kbit_training` + `get_peft_model` for you — it does exactly what you did by hand. Restart the kernel first to clear the reserved pool from the previous notebook.

In [9]:
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [10]:
model = AutoModelForCausalLM.from_pretrained(
    tool_forge.QWEN_4B_BASE,
    quantization_config=nf4_config,
    dtype=torch.bfloat16,
    device_map={"":0}
    )

Loading weights:   0%|          | 1/398 [00:00<01:12,  5.47it/s]/home/sjullens/Code/Projects/LLM_structured_response/.venv-train/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 398/398 [00:02<00:00, 192.25it/s]


In [11]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    target_modules="all-linear",
    lora_alpha=32,
    lora_dropout=0.05,
)

In [12]:
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
cast(PeftModel, model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

## Configure the SFT run (`SFTConfig`)
Smoke-test settings — small and fast, just enough to prove the machinery.

- `output_dir="out/sft-smoke"`
- `max_length=1024` (from the length probe; >1024 truncates ~3%)
- `per_device_train_batch_size=1`, `gradient_accumulation_steps=4`
- `gradient_checkpointing=True` (trade compute for VRAM)
- `bf16=True`
- `learning_rate=2e-4` (typical for QLoRA adapters; tune later)
- `max_steps=30` (smoke only — delete for a real run)
- `logging_steps=1` (watch every step's loss)
- `report_to="none"` (no W&B for a smoke test)
- leave `completion_only_loss` **unset** → it auto-enables on prompt-completion data

In [13]:
sft_config = SFTConfig(
    "out/sft-smoke",
    max_length=1024,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-4,
    max_steps=30,
    logging_steps=1,
    report_to="none",
)

## Build the trainer (`SFTTrainer`)
- `model=` your peft model (or base id + `peft_config=lora_config`)
- `args=sft_config`
- `train_dataset=` the mapped dataset
- `processing_class=tokenizer`
- pass `peft_config` **only** if `model` is a base id; omit it if you handed in a pre-wrapped peft model (else it double-wraps)

In [14]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer
)

Building labels for train dataset: 100%|██████████| 256/256 [00:00<00:00, 2579.13 examples/s]


## Verify the label mask on one batch
Before training, pull one collated batch and confirm the boundary holds at the **token** level — the proof we deferred from the probe.

- `batch = next(iter(trainer.get_train_dataloader()))`
- `labels` should be `-100` on the prompt span and real token ids on the completion
- decode the non-`-100` labels and confirm they are the `<tool_call>` JSON, **not** the user query or the tool schemas
- if the whole row is `-100`, masking/boundary is broken → stop before wasting a training run

In [15]:
# Get a single batch item (batch_size=1)
batch = next(iter(trainer.get_train_dataloader()))

In [16]:
# batch = {"input_ids": Tensor, "attention_mask": Tensor, "labels": Tensor}
input_ids = batch["input_ids"]
labels = batch["labels"]

In [17]:
# Get the input ids whose labels are unmasked 
unmasked_labels = labels != -100

In [18]:
# Filters tokens to indices where True
unmasked_inputs = input_ids[unmasked_labels]
len(unmasked_inputs)

30

In [19]:
# Decode ids into strings. We are expecting only the tool call
tokenizer.decode(unmasked_inputs)

'<tool_call>\n{"name": "getallquotes", "arguments": {"limit": 15, "page": 2}}\n</tool_call><|im_end|>\n<|im_end|>'

## Smoke-test: train ~30 steps
- `trainer.train()`
- watch the loss: it must be **finite** (not nan/inf) and **trend down** across the 30 steps
- flat, rising, or nan loss = template / masking / LR bug — stop and debug, do not scale up

In [20]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,1.248999
2,1.232584
3,1.243171
4,1.177637
5,0.836239
6,0.690370
7,0.655727
8,0.899786
9,0.702657
10,1.004973


TrainOutput(global_step=30, training_loss=0.8826056977113088, metrics={'train_runtime': 88.6221, 'train_samples_per_second': 1.354, 'train_steps_per_second': 0.339, 'total_flos': 1247027626828800.0, 'train_loss': 0.8826056977113088, 'epoch': 0.46875})

## Confirm it fit
- `torch.cuda.max_memory_allocated() / 1e9` — peak allocated during training
- compare to the 12 GB ceiling; if it OOM'd, the levers in order: gradient checkpointing (on?), lower `max_length`, then free WSL display memory

In [21]:
print(f"{torch.cuda.max_memory_allocated() / 1e9} GB")

4.985129984 GB
